# RL Trading Agent — Portfolio Allocation
## Reinforcement Learning Project · CSC_52081_EP

**Self-contained notebook** — every dependency, data pipeline, agent and evaluation step is inlined.

### Agents
| # | Agent | Family | Key idea |
|---|-------|--------|----------|
| 1 | **Exp3** | Adversarial bandit | No stationarity assumption; exploration via exponential weights |
| 2 | **REINFORCE** | Policy gradient | Monte-Carlo returns; on-policy |
| 3 | **PPO** | Policy gradient | Clipped surrogate objective; on-policy; actor-critic |

### Baselines
| Baseline | Description |
|----------|-------------|
| **Buy & Hold** | Equal-weight, never rebalance |
| **DDPG (SB3)** | Off-policy actor-critic — FinRL standard for continuous portfolio allocation |

### Universe & split
| | |
|--|--|
| Stocks | AAPL (tech), JPM (finance), XOM (energy) |
| Train | 2019 – 2021 (bull market) |
| Test | 2022 – 2023 (bear market, out-of-sample) |
| State | 7 × 3 matrix: rolling 252-day covariance + 4 technical indicators |
| Action | Portfolio weight logits → softmax inside `StockPortfolioEnv` |


## 1 · Installation
Uncomment and run **once**, then comment back out.


In [ ]:
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git
# !pip install stable-baselines3[extra] yfinance stockstats torch gymnasium


## 2 · Imports


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Dirichlet
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.meta.env_portfolio_allocation.env_portfolio import StockPortfolioEnv

warnings.filterwarnings('ignore')
os.makedirs('results', exist_ok=True)   # FinRL saves plots here at episode end


## 3 · Configuration
Single source of truth for all constants and hyper-parameters.


In [ ]:
STOCKS           = ['AAPL', 'JPM', 'XOM']

TRAIN_START      = '2018-01-01'   # download start (252-day lookback for covariance)
TRAIN_END        = '2021-12-31'
TEST_START       = '2022-01-01'
TEST_END         = '2023-12-31'

INITIAL_CAPITAL  = 100_000
TRANSACTION_COST = 0.001          # 0.1 %
REWARD_SCALING   = 1e-4

INDICATORS       = ['macd', 'rsi_30', 'cci_30', 'dx_30']

RANDOM_SEED      = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


## 4 · Environment Setup

Pipeline:
1. Download OHLCV data from Yahoo Finance via FinRL
2. Compute 4 technical indicators (MACD, RSI, CCI, DX) via `stockstats`
3. Add a **252-day rolling covariance matrix** (3×3) per date — standard in finance
4. Split train / test
5. Wrap in `StockPortfolioEnv`

**Why bull-market train / bear-market test?**  
Strict out-of-sample evaluation: agents trained in a favourable regime must generalise
to a hostile one. Measures robustness, not just optimisation.


In [ ]:
def get_data():
    df = YahooDownloader(
        start_date=TRAIN_START, end_date=TEST_END, ticker_list=STOCKS
    ).fetch_data()
    return df


def add_features(df):
    fe = FeatureEngineer(
        use_technical_indicator=True,
        tech_indicator_list=INDICATORS,
        use_turbulence=False,
        user_defined_feature=False
    )
    df = fe.preprocess_data(df)
    return df.ffill().fillna(0)


def add_covariance(df, lookback=252):
    """
    252-day rolling covariance matrix (3x3) per date.
    First `lookback` dates dropped (not enough history).
    """
    df = df.sort_values(['date', 'tic']).reset_index(drop=True)
    df.index = df.date.factorize()[0]
    unique_dates = df.date.unique()
    cov_list = []
    for i in range(lookback, len(unique_dates)):
        prices  = df.loc[i - lookback:i].pivot_table(index='date', columns='tic', values='close')
        returns = prices.pct_change().dropna()
        cov_list.append(returns.cov().values)
    df_cov = pd.DataFrame({'date': unique_dates[lookback:], 'cov_list': cov_list})
    df = df.merge(df_cov, on='date')
    return df.sort_values(['date', 'tic']).reset_index(drop=True)


def split_data(df):
    train = df[df.date < TEST_START].reset_index(drop=True)
    test  = df[df.date >= TEST_START].reset_index(drop=True)
    print(f'Train : {train.date.min()} -> {train.date.max()}  ({len(train)} rows)')
    print(f'Test  : {test.date.min()} -> {test.date.max()}  ({len(test)} rows)')
    return train, test


def make_env(df):
    """
    StockPortfolioEnv:
      action  = weight logits [w_AAPL, w_JPM, w_XOM]  -> softmax internally
      state   = 7x3 matrix (covariance + indicators)
      reward  = raw portfolio value
    """
    stock_dim = len(STOCKS)
    df = df.sort_values(['date', 'tic']).reset_index(drop=True)
    df.index = df.date.factorize()[0]
    return StockPortfolioEnv(
        df=df, stock_dim=stock_dim, hmax=100,
        initial_amount=INITIAL_CAPITAL,
        transaction_cost_pct=TRANSACTION_COST,
        reward_scaling=REWARD_SCALING,
        state_space=stock_dim, action_space=stock_dim,
        tech_indicator_list=INDICATORS, turbulence_threshold=None,
    )


In [ ]:
print('Step 1 — Downloading data...')
df_raw  = get_data()
print('Step 2 — Adding technical indicators...')
df_feat = add_features(df_raw)
print('Step 3 — Computing rolling covariance...')
df_cov  = add_covariance(df_feat)
print('Step 4 — Splitting...')
train_df, test_df = split_data(df_cov)
print('Step 5 — Creating environments...')
train_env = make_env(train_df)
test_env  = make_env(test_df)
print('Done.')


## 5 · Agent 1 — Exp3 (Adversarial Bandit)

### Motivation
UCB and Thompson Sampling assume a **stationary** reward distribution.
Financial markets are **non-stationary** (regime changes, macro events).
**Exp3 makes no stationarity assumption** — it is designed for adversarial environments
where the reward distribution can change arbitrarily over time.

### Design choices
| Choice | Value | Reason |
|--------|-------|--------|
| Arms K | 3 | One per stock |
| Action | 100% in selected stock | Simplest, interpretable allocation |
| State | **None** (stateless bandit) | Intentional: comparison point vs state-aware agents |
| Reward | Daily return ∈ [-5%,+5%] → [0,1] | Exp3 theory requires r ∈ [0,1] |

Comparing Exp3 (stateless) vs REINFORCE/PPO (state-aware) directly measures
**how much the state features (covariance + indicators) actually help**.

### Algorithm
```
p_i = (1-γ) · w_i / Σw  +  γ/K     # mixed strategy (exploration guaranteed)
r̂_i = r / p_i                        # importance-weighted estimator (unbiased)
w_i ← w_i · exp(γ · r̂_i / K)        # exponential weight update
```


In [ ]:
class Exp3Agent:
    """
    Exp3 adversarial bandit for portfolio allocation.

    Parameters
    ----------
    n_arms : int   — arms = stocks (default 3)
    gamma  : float — exploration param in (0,1]; typical 0.05-0.2
    seed   : int
    """

    def __init__(self, n_arms=3, gamma=0.1, seed=RANDOM_SEED):
        self.n_arms  = n_arms
        self.gamma   = gamma
        self.rng     = np.random.default_rng(seed)
        self.weights = np.ones(n_arms, dtype=np.float64)  # uniform init
        self.asset_memory = []; self.return_memory = []; self.action_memory = []

    # ── Core Exp3 ─────────────────────────────────────────────────────────────

    def _get_probabilities(self):
        """p_i = (1-γ)·w_i/Σw + γ/K  — guarantees p_i > 0 always."""
        return (1. - self.gamma) * (self.weights / self.weights.sum()) + self.gamma / self.n_arms

    def select_arm(self):
        probs = self._get_probabilities()
        return int(self.rng.choice(self.n_arms, p=probs)), probs

    def update(self, arm, reward, probs):
        """
        Importance-weighted update (only played arm):
            r_hat = reward / p_arm
            w_arm <- w_arm * exp(gamma * r_hat / K)
        """
        self.weights[arm] *= np.exp(self.gamma * (reward / probs[arm]) / self.n_arms)
        self.weights = np.clip(self.weights, 1e-300, 1e300)

    # ── Helpers ───────────────────────────────────────────────────────────────

    def _arm_to_action(self, arm):
        """
        Large logit -> softmax inside env gives ~100% to selected stock.
        [10, 0, 0] -> softmax -> [~0.9999, ~0.0001, ~0.0001]
        """
        a = np.zeros(self.n_arms, dtype=np.float32); a[arm] = 10.; return a

    @staticmethod
    def _normalise_reward(new_v, old_v):
        """
        Map daily return in [-5%, +5%] linearly to [0, 1].
        Formula: (daily_return + 0.05) / 0.10
        Exp3 theory assumes r in [0,1] for the estimator to be unbiased.
        """
        return float(np.clip(((new_v - old_v) / old_v + 0.05) / 0.10, 0., 1.))

    # ── Training ──────────────────────────────────────────────────────────────

    def train(self, env, n_episodes=5):
        """
        n_episodes full passes through training data.
        Weights NOT reset between episodes -> knowledge accumulates.
        """
        self.weights = np.ones(self.n_arms, dtype=np.float64)
        for ep in range(n_episodes):
            state, _ = env.reset()
            done = False; prev_v = float(env.portfolio_value)
            while not done:
                arm, probs = self.select_arm()
                _, reward, terminated, truncated, _ = env.step(self._arm_to_action(arm))
                done = terminated or truncated; new_v = float(reward)
                self.update(arm, self._normalise_reward(new_v, prev_v), probs)
                prev_v = new_v
            print(f'[Exp3] ep {ep+1}/{n_episodes}  value={new_v:,.2f}  weights={self.weights.round(3)}')

    # ── Evaluation ────────────────────────────────────────────────────────────

    def test(self, env):
        """Evaluate on test env. Weights frozen."""
        self.asset_memory = []; self.return_memory = []; self.action_memory = []
        state, _ = env.reset()
        done = False; pv = float(env.portfolio_value); self.asset_memory.append(pv)
        while not done:
            arm, _ = self.select_arm()
            action = self._arm_to_action(arm); self.action_memory.append(action.copy())
            _, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated; nv = float(reward)
            self.return_memory.append((nv - pv) / pv)
            self.asset_memory.append(nv); pv = nv
        return self.get_results()

    def _compute_metrics(self):
        r = np.array(self.return_memory)
        tr = (self.asset_memory[-1] - self.asset_memory[0]) / self.asset_memory[0]
        sh = float((252**.5) * r.mean() / r.std()) if r.std() > 0 else 0.
        return float(tr), sh

    def get_results(self):
        tr, sh = self._compute_metrics()
        return {'portfolio_values': self.asset_memory, 'daily_returns': self.return_memory,
                'actions': self.action_memory, 'total_return': tr, 'sharpe': sh}


In [ ]:
exp3_agent = Exp3Agent(n_arms=3, gamma=0.1, seed=RANDOM_SEED)
exp3_agent.train(train_env, n_episodes=5)


In [ ]:
exp3_results = exp3_agent.test(test_env)
print(f"Exp3  total return : {exp3_results['total_return']:+.2%}")
print(f"Exp3  Sharpe ratio : {exp3_results['sharpe']:.4f}")


## 6 · Agent 2 — REINFORCE

> **Section completed by teammate.**

### Algorithm
REINFORCE is a Monte-Carlo policy gradient method:
- Policy π_θ(a|s): neural network → **Dirichlet** distribution over portfolio weights
  (naturally constrained to the simplex: Σw=1, w≥0)
- Returns G_t = Σ_k γ^k · r_{t+k} computed at episode end
- Gradient: ∇J(θ) = E[G_t · ∇ log π_θ(a_t|s_t)]
- Baseline: standardise returns to reduce variance

State flattened to 21 features before being fed to the network.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# REINFORCE — to be filled in by teammate
# Expected interface:
#   reinforce_agent = ReinforceAgent(state_dim=21, n_assets=3, lr=1e-3, gamma=0.99)
#   reinforce_agent.train(train_env, n_episodes=200)
#   reinforce_results = reinforce_agent.test(test_env)
#   reinforce_results keys: portfolio_values, daily_returns, actions, total_return, sharpe
# ─────────────────────────────────────────────────────────────────────────────

# class PolicyNetwork(nn.Module): ...
# class ReinforceAgent: ...

reinforce_results = None   # replace when implemented


## 7 · Agent 3 — PPO (Proximal Policy Optimisation)

> **Section completed by teammate (builds on REINFORCE).**

### How PPO improves on REINFORCE
| REINFORCE | PPO |
|-----------|-----|
| One gradient step per episode | Multiple mini-batch updates per rollout |
| No constraint on update size | Clipped surrogate: `min(r·A, clip(r,1-ε,1+ε)·A)` |
| No value baseline | Actor-critic: shared backbone + separate V(s) head |
| High variance returns | GAE (λ=0.95) for lower-variance advantage estimates |

Same Dirichlet policy as REINFORCE. Clip ε = 0.2 by default.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PPO — to be filled in by teammate
# Expected interface:
#   ppo_agent = PPOAgent(state_dim=21, n_assets=3, lr=3e-4, gamma=0.99,
#                        gae_lambda=0.95, clip_eps=0.2, n_epochs=4)
#   ppo_agent.train(train_env, n_episodes=200)
#   ppo_results = ppo_agent.test(test_env)
# ─────────────────────────────────────────────────────────────────────────────

# class ActorCritic(nn.Module): ...
# class PPOAgent: ...

ppo_results = None   # replace when implemented


## 8 · Baselines

> **Section completed by teammate.**

### Buy & Hold
Equal-weight portfolio [1/3, 1/3, 1/3], allocated once and never rebalanced.
Natural lower bound — zero adaptation to market signals.

### DDPG (Stable-Baselines3)
Off-policy actor-critic with deterministic policy + action noise for exploration.
**Why DDPG as baseline?** FinRL's standard algorithm for continuous portfolio allocation;
provides a learned competitive reference without implementing it from scratch.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Baselines — to be filled in by teammate
# Expected:
#   bnh_results  = run_buy_and_hold(test_env)
#   ddpg_results = run_ddpg(train_env, test_env, total_timesteps=50_000)
# ─────────────────────────────────────────────────────────────────────────────

# def run_buy_and_hold(env): ...
# def run_ddpg(train_env, test_env, total_timesteps): ...

bnh_results  = None   # replace when implemented
ddpg_results = None   # replace when implemented


## 9 · Results Comparison

Metrics on the **2022-2023 bear market test set**:
- **Total return** = (final value − initial value) / initial value
- **Sharpe ratio** = √252 × mean(daily_returns) / std(daily_returns) — annualised risk-adjusted return


In [ ]:
# Collect results — None entries are skipped (not yet implemented)
all_results = {
    'Exp3'       : exp3_results,
    'REINFORCE'  : reinforce_results,
    'PPO'        : ppo_results,
    'Buy & Hold' : bnh_results,
    'DDPG'       : ddpg_results,
}
all_results = {k: v for k, v in all_results.items() if v is not None}

print(f"{'Agent':<14} {'Total Return':>14} {'Sharpe':>10}")
print('-' * 40)
for name, res in all_results.items():
    print(f"{name:<14} {res['total_return']:>+13.2%} {res['sharpe']:>10.4f}")


In [ ]:
# Portfolio value curves
COLORS = {'Exp3':'#2196F3','REINFORCE':'#4CAF50','PPO':'#FF9800',
          'Buy & Hold':'#9E9E9E','DDPG':'#E91E63'}
STYLES = {'Buy & Hold':'--','DDPG':'--'}

fig, ax = plt.subplots(figsize=(12, 6))
for name, res in all_results.items():
    ax.plot(res['portfolio_values'],
            label=f"{name}  ({res['total_return']:+.1%} | Sharpe {res['sharpe']:.2f})",
            color=COLORS.get(name,'k'), linestyle=STYLES.get(name,'-'), lw=1.8)

ax.axhline(INITIAL_CAPITAL, color='black', linewidth=0.6, linestyle=':', label='Initial capital')
ax.set_title('Portfolio Value — Test 2022-2023 (Bear Market)', fontsize=14)
ax.set_xlabel('Trading days'); ax.set_ylabel('Portfolio value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/portfolio_comparison.png', dpi=150)
plt.show()


In [ ]:
# Sharpe ratio bar chart
names   = list(all_results.keys())
sharpes = [all_results[n]['sharpe'] for n in names]
colors  = [COLORS.get(n, 'grey') for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, sharpes, color=colors, edgecolor='white', lw=0.5)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Annualised Sharpe Ratio — Test 2022-2023', fontsize=13)
ax.set_ylabel('Sharpe Ratio')
for bar, val in zip(bars, sharpes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('results/sharpe_comparison.png', dpi=150)
plt.show()


## 10 · Reward Ablation — Exp3

**Research question:** how sensitive is Exp3 to the reward normalisation choice?

Exp3 theory requires r ∈ [0, 1] for the importance-weighted estimator to be unbiased.
We test three definitions:

| Variant | Formula | Comment |
|---------|---------|----------|
| **Default** `[-5%, +5%] → [0, 1]` | `(r + 0.05) / 0.10` | Theoretically correct |
| **Wider clip** `[-10%, +10%] → [0, 1]` | `(r + 0.10) / 0.20` | More tolerance for large moves |
| **Raw value** (naïve) | `value / 100_000` | Outside [0,1] — theory violated |

This ablation shows why reward design matters even for simple bandit algorithms.


In [ ]:
def run_exp3_variant(train_env, test_env, reward_fn, label, gamma=0.1):
    """Train and evaluate Exp3 with a custom reward function."""

    class VariantExp3(Exp3Agent):
        @staticmethod
        def _normalise_reward(nv, ov): return reward_fn(nv, ov)

    agent = VariantExp3(n_arms=3, gamma=gamma, seed=RANDOM_SEED)
    agent.train(train_env, n_episodes=5)
    res = agent.test(test_env)
    print(f'  [{label}]  return={res["total_return"]:+.2%}   Sharpe={res["sharpe"]:.4f}')
    return res


reward_variants = {
    'Default  [-5%,+5%]'  : lambda nv, ov: float(np.clip(((nv-ov)/ov + 0.05) / 0.10, 0, 1)),
    'Wider  [-10%,+10%]'  : lambda nv, ov: float(np.clip(((nv-ov)/ov + 0.10) / 0.20, 0, 1)),
    'Raw value (naive)'   : lambda nv, ov: float(np.clip(nv / INITIAL_CAPITAL,        0, 2)),
}

print('Exp3 reward ablation:')
ablation_results = {
    lbl: run_exp3_variant(train_env, test_env, fn, lbl)
    for lbl, fn in reward_variants.items()
}


In [ ]:
# Ablation curves
abl_colors = ['#2196F3', '#FF5722', '#9C27B0']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: portfolio value curves
ax = axes[0]
for (label, res), color in zip(ablation_results.items(), abl_colors):
    ax.plot(res['portfolio_values'],
            label=f"{label}\n({res['total_return']:+.1%} | Sharpe {res['sharpe']:.2f})",
            color=color, lw=1.8)
ax.axhline(INITIAL_CAPITAL, color='black', lw=0.6, linestyle=':')
ax.set_title('Portfolio Value — Reward Ablation', fontsize=12)
ax.set_xlabel('Trading days'); ax.set_ylabel('Portfolio value ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Right: Sharpe bars
ax2 = axes[1]
lbs = list(ablation_results.keys())
shs = [ablation_results[l]['sharpe'] for l in lbs]
bars = ax2.bar(range(len(lbs)), shs, color=abl_colors, edgecolor='white')
ax2.set_xticks(range(len(lbs))); ax2.set_xticklabels(lbs, fontsize=9, rotation=10)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Sharpe Ratio — Reward Ablation', fontsize=12)
ax2.set_ylabel('Sharpe Ratio')
for bar, val in zip(bars, shs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('results/reward_ablation.png', dpi=150)
plt.show()
